# Installing Dependencies

In [1]:
%pip install lightgbm scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load

In [2]:
import pandas as pd

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)

TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)


## 1. Import Libraries

In [3]:
import numpy as np
import lightgbm as lgb
from scipy import stats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_LABEL stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())
print()
print('Train PIDs:', sorted(TRAIN_LABEL['pid'].unique()))
print('Test  PIDs:', sorted(TEST_LABEL['pid'].unique()))
print('Overlap   :', set(TRAIN_LABEL['pid'].unique()) & set(TEST_LABEL['pid'].unique()))
print()
print('NOTE: Zero PID overlap — test participants are all new people.')
print('This means per-person z-score normalisation is critical.')

TRAIN_LABEL stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64

Train PIDs: ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']
Test  PIDs: ['01Z2', '2XO3', 'D1XP', 'NQRB', 'SE4Q', 'SNG7', 'TF0Y', 'Y21H']
Overlap   : set()

NOTE: Zero PID overlap — test participants are all new people.
This means per-person z-score normalisation is critical.


## 2. Per-Person Z-Score Normalisation

**EDA finding**: EDA varies 60x across people (range 0.09–5.72 in train).
Raw absolute sensor values encode *who the person is*, not *how stressed they are*.
We normalise each sensor by that person's own mean and std computed from their
entire sensor recording. For test persons we use their own test sensor data.

**Key change from v1**: `pid_enc` is REMOVED (was #1 feature but useless on new people).

In [4]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def compute_pid_stats(sensor_df):
    """Compute per-person mean and std for each sensor column."""
    d = {}
    for pid, grp in sensor_df.groupby('pid'):
        d[pid] = {}
        for c in SENSOR_COLS:
            v  = grp[c].dropna().values.astype(float)
            mu = float(np.mean(v)) if len(v) > 0 else 0.0
            sd = float(np.std(v))  if len(v) > 0 else 1.0
            d[pid][c + '_mu'] = mu
            d[pid][c + '_sd'] = max(sd, 1e-6)
    return d

train_pid_stats = compute_pid_stats(TRAIN_DATA)
test_pid_stats  = compute_pid_stats(TEST_DATA)

# Global fallback for any unseen PID (median of train per-person stats)
global_mu = {c: np.median([train_pid_stats[p][c+'_mu'] for p in train_pid_stats]) for c in SENSOR_COLS}
global_sd = {c: np.median([train_pid_stats[p][c+'_sd'] for p in train_pid_stats]) for c in SENSOR_COLS}

print('Per-person EDA baseline (train) — shows why absolute values are useless cross-person:')
for pid in sorted(train_pid_stats):
    print(f'  {pid}: eda_mean={train_pid_stats[pid]["eda_mu"]:.3f}, '
          f'temp_mean={train_pid_stats[pid]["temperature_mu"]:.3f}')

print()
print('Per-person EDA baseline (test):')
for pid in sorted(test_pid_stats):
    print(f'  {pid}: eda_mean={test_pid_stats[pid]["eda_mu"]:.3f}, '
          f'temp_mean={test_pid_stats[pid]["temperature_mu"]:.3f}')

Per-person EDA baseline (train) — shows why absolute values are useless cross-person:
  43JW: eda_mean=2.072, temp_mean=33.377
  C8Q6: eda_mean=0.127, temp_mean=29.952
  DT5C: eda_mean=4.531, temp_mean=33.462
  F1ZM: eda_mean=0.094, temp_mean=29.457
  HDS9: eda_mean=5.719, temp_mean=30.171
  P4DZ: eda_mean=1.183, temp_mean=29.567
  TPQI: eda_mean=3.551, temp_mean=32.924

Per-person EDA baseline (test):
  01Z2: eda_mean=0.401, temp_mean=29.864
  2XO3: eda_mean=4.231, temp_mean=34.276
  D1XP: eda_mean=3.376, temp_mean=33.170
  NQRB: eda_mean=0.226, temp_mean=32.009
  SE4Q: eda_mean=3.834, temp_mean=33.795
  SNG7: eda_mean=0.728, temp_mean=33.528
  TF0Y: eda_mean=2.933, temp_mean=32.826
  Y21H: eda_mean=10.214, temp_mean=33.984


## 3. Feature Extraction — Z-scored + Delta + Slope Features

Changes from v1:
- All 11 stats are computed on **z-scored** values (relative to that person's baseline)
- `delta` = mean(last 90s) − mean(first 90s) — captures **within-window change**
- `slope` = linear trend across the full 3-minute window
- `pid_enc` is **not added** (removed, was leaking person identity)

Stress is about *relative change* from a person's own baseline, not absolute levels.

In [5]:
WINDOW_MS = 180_000   # 3 minutes
HALF_MS   =  90_000   # split point for delta features

def extract_features(label_df, sensor_df, pid_stats):
    """Extract z-scored statistical + delta + slope features for each label window."""
    sensor_by_pid = {
        pid: grp.sort_values('timestamp').reset_index(drop=True)
        for pid, grp in sensor_df.groupby('pid')
    }

    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']
        ts  = float(lrow['timestamp'])
        lid = lrow['id']
        feat = {'id': lid}

        if pid not in sensor_by_pid:
            # No sensor data for this PID — all NaN (imputer handles later)
            for c in SENSOR_COLS:
                for s in ['mean','std','min','max','median','skew','kurt',
                          'range','q25','q75','iqr','delta','slope']:
                    feat[f'{c}_{s}'] = np.nan
            feat.update({'accel_mag_mean': np.nan, 'accel_mag_std': np.nan,
                         'accel_mag_max': np.nan})
            rows.append(feat)
            continue

        sg     = sensor_by_pid[pid]
        ts_arr = sg['timestamp'].values

        # Full 3-min window, first-half and last-half
        win_all   = sg.loc[(ts_arr >= ts - WINDOW_MS) & (ts_arr <= ts),      SENSOR_COLS]
        win_first = sg.loc[(ts_arr >= ts - WINDOW_MS) & (ts_arr <  ts - HALF_MS), SENSOR_COLS]
        win_last  = sg.loc[(ts_arr >= ts - HALF_MS)   & (ts_arr <= ts),      SENSOR_COLS]

        for c in SENSOR_COLS:
            mu = pid_stats.get(pid, {}).get(c + '_mu', global_mu[c])
            sd = pid_stats.get(pid, {}).get(c + '_sd', global_sd[c])

            v_all   = (win_all[c].dropna().values.astype(float)   - mu) / sd
            v_first = (win_first[c].dropna().values.astype(float) - mu) / sd
            v_last  = (win_last[c].dropna().values.astype(float)  - mu) / sd

            if len(v_all) == 0:
                for s in ['mean','std','min','max','median','skew','kurt',
                          'range','q25','q75','iqr','delta','slope']:
                    feat[f'{c}_{s}'] = np.nan
                continue

            # Standard stats (on z-scored values)
            feat[f'{c}_mean']   = float(np.mean(v_all))
            feat[f'{c}_std']    = float(np.std(v_all))
            feat[f'{c}_min']    = float(np.min(v_all))
            feat[f'{c}_max']    = float(np.max(v_all))
            feat[f'{c}_median'] = float(np.median(v_all))
            feat[f'{c}_skew']   = float(stats.skew(v_all))     if len(v_all) > 2 else 0.0
            feat[f'{c}_kurt']   = float(stats.kurtosis(v_all)) if len(v_all) > 2 else 0.0
            feat[f'{c}_range']  = float(np.max(v_all) - np.min(v_all))
            feat[f'{c}_q25']    = float(np.percentile(v_all, 25))
            feat[f'{c}_q75']    = float(np.percentile(v_all, 75))
            feat[f'{c}_iqr']    = float(np.percentile(v_all, 75) - np.percentile(v_all, 25))

            # Delta: last-half mean minus first-half mean (sign encodes direction)
            if len(v_first) > 0 and len(v_last) > 0:
                feat[f'{c}_delta'] = float(np.mean(v_last) - np.mean(v_first))
            else:
                feat[f'{c}_delta'] = 0.0

            # Slope: linear trend across the window
            if len(v_all) > 2:
                t_norm = np.linspace(0, 1, len(v_all))
                feat[f'{c}_slope'] = float(np.polyfit(t_norm, v_all, 1)[0])
            else:
                feat[f'{c}_slope'] = 0.0

        # Accel magnitude (not z-scored — magnitude is always positive)
        ax = win_all['accel_x'].values
        ay = win_all['accel_y'].values
        az = win_all['accel_z'].values
        if len(ax) > 0:
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = np.nan
            feat['accel_mag_std']  = np.nan
            feat['accel_mag_max']  = np.nan

        rows.append(feat)

    return pd.DataFrame(rows).set_index('id')


print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_stats)
print(f'  shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, test_pid_stats)
print(f'  shape: {test_features.shape}')

Extracting train features...
  shape: (815, 81)
Extracting test features...
  shape: (1028, 81)


## 4. Prepare X, y — No pid_enc

In [6]:
train_label_indexed = TRAIN_LABEL.set_index('id')
test_label_indexed  = TEST_LABEL.set_index('id')

# NOTE: pid_enc intentionally NOT added.
# EDA showed it was #1 LightGBM feature but encodes person identity,
# which causes total breakdown on the 8 unseen test participants.
X      = train_features.copy()
X_test = test_features.copy()

y      = train_label_indexed['stress'].astype(int)
groups = train_label_indexed['pid']   # used for LOPO CV

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(X),      columns=X.columns,      index=X.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test),     columns=X_test.columns, index=X_test.index)

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class distribution:', y.value_counts().sort_index().to_dict())
print('Features added   : delta + slope (13 stats per sensor, was 11)')
print('Features removed : pid_enc')

X_imp shape     : (815, 81)
X_test_imp shape: (1028, 81)
Class distribution: {0: 162, 1: 66, 2: 587}
Features added   : delta + slope (13 stats per sensor, was 11)
Features removed : pid_enc


## 5. Leave-One-Person-Out CV (True Generalization Estimate)

**Why LOPO instead of standard 5-fold?**
Standard StratifiedKFold mixes data from the same person across train/val splits.
This leaks person-specific patterns (body temperature, EDA baseline) into validation,
giving optimistic scores that don't reflect cross-person generalization.

LOPO holds out one entire person at a time — mimicking the actual test condition
where all 8 test participants are completely new (zero overlap with train PIDs).
Use LOPO score as your primary metric, not standard CV.

In [7]:
counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {c: total / (n_cls * cnt) for c, cnt in counts.items()}
sample_weights = np.array([class_weights[yi] for yi in y])
print('Class weights:', {k: round(v,3) for k,v in class_weights.items()})

LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.02,
    num_leaves        = 63,
    max_depth         = -1,
    min_child_samples = 5,
    subsample         = 0.7,
    colsample_bytree  = 0.7,
    reg_alpha         = 0.3,
    reg_lambda        = 0.3,
    class_weight      = 'balanced',
    objective         = 'multiclass',
    num_class         = 3,
    n_jobs            = -1,
    verbose           = -1,
)

print('\n=== Leave-One-Person-Out CV ===')
logo       = LeaveOneGroupOut()
lopo_scores = []

for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[val_idx[0]]
    y_val   = y.iloc[val_idx]

    if len(y_val.unique()) < 2:
        print(f'  Skip {pid_val}: only class {y_val.unique()} present')
        continue

    m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
    m.fit(
        X_imp.iloc[tr_idx], y.iloc[tr_idx],
        sample_weight = sample_weights[tr_idx],
        eval_set      = [(X_imp.iloc[val_idx], y_val)],
        callbacks     = [lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)],
    )
    sc = balanced_accuracy_score(y_val, m.predict(X_imp.iloc[val_idx]))
    print(f'  Leave out {pid_val}: {sc:.4f}  '
          f'(n={len(val_idx)}, classes={sorted(y_val.unique())})')
    lopo_scores.append(sc)

print(f'\nLOPO CV = {np.mean(lopo_scores):.4f} \u00b1 {np.std(lopo_scores):.4f}')
print('This is your REAL generalization estimate. Use this, not standard 5-fold CV.')

Class weights: {1: 4.116, 0: 1.677, 2: 0.463}

=== Leave-One-Person-Out CV ===
  Leave out 43JW: 0.2308  (n=93, classes=[np.int64(0), np.int64(2)])
  Leave out C8Q6: 0.4401  (n=152, classes=[np.int64(0), np.int64(2)])
  Leave out DT5C: 0.3991  (n=90, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out F1ZM: 0.4552  (n=137, classes=[np.int64(1), np.int64(2)])
  Leave out HDS9: 0.6966  (n=135, classes=[np.int64(0), np.int64(2)])
  Leave out P4DZ: 0.2994  (n=144, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out TPQI: 0.4773  (n=64, classes=[np.int64(0), np.int64(2)])

LOPO CV = 0.4284 ± 0.1372
This is your REAL generalization estimate. Use this, not standard 5-fold CV.


## 6. Standard 5-Fold CV (For Reference Only)

Included only to compare against LOPO. Do NOT use this to judge model quality.
If standard CV >> LOPO, the model is overfitting to person identity.

In [8]:
SEEDS = [42, 7, 123]
std_cv_all = []

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_scores = []
    for tr_idx, val_idx in skf.split(X_imp, y):
        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m.fit(
            X_imp.iloc[tr_idx], y.iloc[tr_idx],
            sample_weight = sample_weights[tr_idx],
            eval_set      = [(X_imp.iloc[val_idx], y.iloc[val_idx])],
            callbacks     = [lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)],
        )
        fold_scores.append(balanced_accuracy_score(y.iloc[val_idx], m.predict(X_imp.iloc[val_idx])))
    ms = np.mean(fold_scores)
    std_cv_all.append(ms)
    print(f'Seed {seed}: Standard CV = {ms:.4f} \u00b1 {np.std(fold_scores):.4f}')

print(f'\nMean Standard CV = {np.mean(std_cv_all):.4f}')
print(f'LOPO CV          = {np.mean(lopo_scores):.4f}')
gap = np.mean(std_cv_all) - np.mean(lopo_scores)
print(f'Gap (overfitting) = {gap:.4f}  \u2190 should be smaller than v1 (was ~0.47)')

Seed 42: Standard CV = 0.8043 ± 0.0360
Seed 7: Standard CV = 0.8192 ± 0.0250
Seed 123: Standard CV = 0.8135 ± 0.0492

Mean Standard CV = 0.8123
LOPO CV          = 0.4284
Gap (overfitting) = 0.3840  ← should be smaller than v1 (was ~0.47)


## 7. Final Multi-Seed Ensemble

Train final models using all training data, then soft-vote across 3 seeds.

In [9]:
print('Training final ensemble...')
all_test_proba = []
all_cv_scores  = []

for seed in SEEDS:
    skf            = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    test_proba_acc = np.zeros((len(X_test_imp), 3))
    fold_scores    = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr, X_val = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx],     y.iloc[val_idx]
        sw_tr        = sample_weights[tr_idx]

        model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        model.fit(
            X_tr, y_tr,
            sample_weight = sw_tr,
            eval_set      = [(X_val, y_val)],
            callbacks     = [lgb.early_stopping(100, verbose=False),
                             lgb.log_evaluation(-1)],
        )

        val_preds = model.predict(X_val)
        score     = balanced_accuracy_score(y_val, val_preds)
        fold_scores.append(score)
        test_proba_acc += model.predict_proba(X_test_imp)

    test_proba_acc /= 5
    all_test_proba.append(test_proba_acc)
    mean_s = np.mean(fold_scores)
    all_cv_scores.append(mean_s)
    print(f'Seed {seed}: CV = {mean_s:.4f} \u00b1 {np.std(fold_scores):.4f}')

print(f'\nEnsemble CV: {np.mean(all_cv_scores):.4f}')

final_proba = np.mean(all_test_proba, axis=0)
final_preds = np.argmax(final_proba, axis=1).astype(int)

print('\nPrediction distribution:')
unique, counts_pred = np.unique(final_preds, return_counts=True)
for u, c in zip(unique, counts_pred):
    print(f'  class {u}: {c}')

Training final ensemble...
Seed 42: CV = 0.8043 ± 0.0360
Seed 7: CV = 0.8192 ± 0.0250
Seed 123: CV = 0.8135 ± 0.0492

Ensemble CV: 0.8123

Prediction distribution:
  class 0: 130
  class 1: 49
  class 2: 849


# Generating Final Submission

In [10]:
submission = pd.DataFrame({
    'id':     TEST_LABEL['id'].values,
    'stress': final_preds,
})

submission.to_csv('submission_v2.csv', index=False)
print('submission_v2.csv saved!')
print(submission.head(10))

submission_v2.csv saved!
     id  stress
0  1227       1
1  1228       2
2  1229       2
3  1230       2
4  1231       0
5  1232       2
6  1233       2
7  1234       2
8  1235       2
9  1236       2
